## **Cleaning + Data Quality Checks + Transformations**

### **insurance,json (raw file-Bronze zone) file upload**

In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName("Insurance").getOrCreate()
df_raw = spark.read.option("multiline", "true").option("header","true").option("inferschema","true").json("/Volumes/workspace/insurance/insurance_volume/insurance.json")

In [0]:
#Extract the array
df = df_raw.selectExpr("explode(insurance) as record")

In [0]:
df.display()

record
"List(18500, 2024-01-15, C001, Approved, Auto, CU101, 2023-02-10, 14500)"
"List(42000, 2024-01-28, C002, Pending, Health, CU102, 2023-03-05, 22000)"
"List(9800, 2024-02-10, C003, Rejected, Property, CU103, 2023-04-12, 12000)"
"List(26500, 2024-02-22, C004, Approved, Auto, CU104, 2023-05-18, 18000)"
"List(15000, 2024-03-06, C005, Pending, Health, CU105, 2023-06-09, 16000)"
"List(36500, 2024-03-19, C006, Approved, Property, CU106, 2023-07-21, 24000)"
"List(7200, 2024-04-03, C007, Rejected, Auto, CU107, 2023-08-14, 11000)"
"List(49800, 2024-04-18, C008, Approved, Health, CU108, 2023-09-02, 29000)"
"List(31000, 2024-05-01, C009, Pending, Property, CU109, 2023-10-11, 21000)"
"List(22500, 2024-05-16, C010, Approved, Auto, CU110, 2023-11-06, 17000)"


In [0]:
#Flatten the struct

df_final = df.select("record.*")

In [0]:
df_final.display()

claim_amount,claim_date,claim_id,claim_status,claim_type,customer_id,policy_date,premium
18500,2024-01-15,C001,Approved,Auto,CU101,2023-02-10,14500
42000,2024-01-28,C002,Pending,Health,CU102,2023-03-05,22000
9800,2024-02-10,C003,Rejected,Property,CU103,2023-04-12,12000
26500,2024-02-22,C004,Approved,Auto,CU104,2023-05-18,18000
15000,2024-03-06,C005,Pending,Health,CU105,2023-06-09,16000
36500,2024-03-19,C006,Approved,Property,CU106,2023-07-21,24000
7200,2024-04-03,C007,Rejected,Auto,CU107,2023-08-14,11000
49800,2024-04-18,C008,Approved,Health,CU108,2023-09-02,29000
31000,2024-05-01,C009,Pending,Property,CU109,2023-10-11,21000
22500,2024-05-16,C010,Approved,Auto,CU110,2023-11-06,17000


In [0]:
df_final.printSchema()

root
 |-- claim_amount: long (nullable = true)
 |-- claim_date: string (nullable = true)
 |-- claim_id: string (nullable = true)
 |-- claim_status: string (nullable = true)
 |-- claim_type: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- policy_date: string (nullable = true)
 |-- premium: long (nullable = true)



## **Data Cleaning**

In [0]:
# Check exact row duplicates 
df_final.groupBy(df_final.columns).count().filter("count > 1").show()


+------------+----------+--------+------------+----------+-----------+-----------+-------+-----+
|claim_amount|claim_date|claim_id|claim_status|claim_type|customer_id|policy_date|premium|count|
+------------+----------+--------+------------+----------+-----------+-----------+-------+-----+
+------------+----------+--------+------------+----------+-----------+-----------+-------+-----+



In [0]:
# Drop exact duplicates 
df_final.dropDuplicates()

DataFrame[claim_amount: bigint, claim_date: string, claim_id: string, claim_status: string, claim_type: string, customer_id: string, policy_date: string, premium: bigint]

In [0]:
# Drop duplicates based on claim_id (business key)
df_final.dropDuplicates(["claim_id"])
df_final.display()

claim_amount,claim_date,claim_id,claim_status,claim_type,customer_id,policy_date,premium
18500,2024-01-15,C001,Approved,Auto,CU101,2023-02-10,14500
42000,2024-01-28,C002,Pending,Health,CU102,2023-03-05,22000
9800,2024-02-10,C003,Rejected,Property,CU103,2023-04-12,12000
26500,2024-02-22,C004,Approved,Auto,CU104,2023-05-18,18000
15000,2024-03-06,C005,Pending,Health,CU105,2023-06-09,16000
36500,2024-03-19,C006,Approved,Property,CU106,2023-07-21,24000
7200,2024-04-03,C007,Rejected,Auto,CU107,2023-08-14,11000
49800,2024-04-18,C008,Approved,Health,CU108,2023-09-02,29000
31000,2024-05-01,C009,Pending,Property,CU109,2023-10-11,21000
22500,2024-05-16,C010,Approved,Auto,CU110,2023-11-06,17000


In [0]:
# Count nulls per column 
from pyspark.sql import functions as F
df_final.select([F.count(F.when(F.col(c).isNull(), c)).alias(c) for c in df_final.columns]).display()

claim_amount,claim_date,claim_id,claim_status,claim_type,customer_id,policy_date,premium
0,0,0,0,0,0,0,0


In [0]:
# Drop rows with null claim_id or customer_id (critical identifiers) 
df_final.na.drop(subset=["claim_id", "customer_id"])

DataFrame[claim_amount: bigint, claim_date: string, claim_id: string, claim_status: string, claim_type: string, customer_id: string, policy_date: string, premium: bigint]

In [0]:
# Fill categorical nulls with defaults 
df_final.na.fill({"claim_status": "Pending",
            "claim_type": "Unknown" })

DataFrame[claim_amount: bigint, claim_date: string, claim_id: string, claim_status: string, claim_type: string, customer_id: string, policy_date: string, premium: bigint]

In [0]:
from pyspark.sql.functions import col, to_date

df_final = df_final \
    .withColumn("claim_amount", col("claim_amount").cast("long")) \
    .withColumn("premium", col("premium").cast("long")) \
    .withColumn("claim_id", col("claim_id").cast("string")) \
    .withColumn("claim_status", col("claim_status").cast("string")) \
    .withColumn("claim_type", col("claim_type").cast("string")) \
    .withColumn("customer_id", col("customer_id").cast("string")) \
    .withColumn("claim_date", to_date(col("claim_date"), "yyyy-MM-dd")) \
    .withColumn("policy_date", to_date(col("policy_date"), "yyyy-MM-dd"))


In [0]:
df_final = (df_final.withColumnRenamed("claim_amount", "Claim_amount") 
             .withColumnRenamed("claim_date","Claim_date")
             .withColumnRenamed("claim_id","Claim_id")
             .withColumnRenamed("claim_status","Claim_status")
             .withColumnRenamed("claim_type","Claim_type")
             .withColumnRenamed("customer_id","Customer_id")
             .withColumnRenamed("policy_date","Policy_date")
             .withColumnRenamed("premium","Premium"))

In [0]:
df_clean = df_final

In [0]:
df_clean.display()

Claim_amount,Claim_date,Claim_id,Claim_status,Claim_type,Customer_id,Policy_date,Premium
18500,2024-01-15,C001,Approved,Auto,CU101,2023-02-10,14500
42000,2024-01-28,C002,Pending,Health,CU102,2023-03-05,22000
9800,2024-02-10,C003,Rejected,Property,CU103,2023-04-12,12000
26500,2024-02-22,C004,Approved,Auto,CU104,2023-05-18,18000
15000,2024-03-06,C005,Pending,Health,CU105,2023-06-09,16000
36500,2024-03-19,C006,Approved,Property,CU106,2023-07-21,24000
7200,2024-04-03,C007,Rejected,Auto,CU107,2023-08-14,11000
49800,2024-04-18,C008,Approved,Health,CU108,2023-09-02,29000
31000,2024-05-01,C009,Pending,Property,CU109,2023-10-11,21000
22500,2024-05-16,C010,Approved,Auto,CU110,2023-11-06,17000


In [0]:
# Save cleaned dataset (Silver Zone) 
df_clean.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable("insurance.insurance_silver")

## **Transformations**

In [0]:
#Transformation 1 : Fraud Detection Flag
df_transform=df_clean.withColumn("Fraud_flag", F.when(F.col("Claim_amount") > F.col("Premium"), 1).otherwise(0))

In [0]:
#Transformation 2 : Claim Ratio & Risk Segmentation
df_transform= df_transform.withColumn("Claim_ratio", F.col("Claim_amount") / F.col("Premium"))
df_transform = df_transform.withColumn("Risk_segment",
    F.when(F.col("Claim_ratio") > 2, "High Risk")
     .when((F.col("Claim_ratio") >= 1) & (F.col("Claim_ratio") <= 2), "Medium Risk")
     .otherwise("Low Risk"))
     


In [0]:
df_transform.display()

Claim_amount,Claim_date,Claim_id,Claim_status,Claim_type,Customer_id,Policy_date,Premium,Fraud_flag,Claim_ratio,Risk_segment
18500,2024-01-15,C001,Approved,Auto,CU101,2023-02-10,14500,1,1.2758620689655173,Medium Risk
42000,2024-01-28,C002,Pending,Health,CU102,2023-03-05,22000,1,1.9090909090909092,Medium Risk
9800,2024-02-10,C003,Rejected,Property,CU103,2023-04-12,12000,0,0.8166666666666667,Low Risk
26500,2024-02-22,C004,Approved,Auto,CU104,2023-05-18,18000,1,1.4722222222222223,Medium Risk
15000,2024-03-06,C005,Pending,Health,CU105,2023-06-09,16000,0,0.9375,Low Risk
36500,2024-03-19,C006,Approved,Property,CU106,2023-07-21,24000,1,1.5208333333333333,Medium Risk
7200,2024-04-03,C007,Rejected,Auto,CU107,2023-08-14,11000,0,0.6545454545454545,Low Risk
49800,2024-04-18,C008,Approved,Health,CU108,2023-09-02,29000,1,1.717241379310345,Medium Risk
31000,2024-05-01,C009,Pending,Property,CU109,2023-10-11,21000,1,1.4761904761904763,Medium Risk
22500,2024-05-16,C010,Approved,Auto,CU110,2023-11-06,17000,1,1.3235294117647058,Medium Risk


In [0]:
#Transformation 3 : Customer Claim Frequency
Claim_freq = df_transform.groupBy("Customer_id").agg(F.count("Claim_id").alias("claim_count"))
df_transform_claimfreq = df_transform.join(Claim_freq, on="Customer_id", how="left")

In [0]:
df_transform_claimfreq.display()

Customer_id,Claim_amount,Claim_date,Claim_id,Claim_status,Claim_type,Policy_date,Premium,Fraud_flag,Claim_ratio,Risk_segment,claim_count
CU101,18500,2024-01-15,C001,Approved,Auto,2023-02-10,14500,1,1.2758620689655173,Medium Risk,1
CU102,42000,2024-01-28,C002,Pending,Health,2023-03-05,22000,1,1.9090909090909092,Medium Risk,1
CU103,9800,2024-02-10,C003,Rejected,Property,2023-04-12,12000,0,0.8166666666666667,Low Risk,1
CU104,26500,2024-02-22,C004,Approved,Auto,2023-05-18,18000,1,1.4722222222222223,Medium Risk,1
CU105,15000,2024-03-06,C005,Pending,Health,2023-06-09,16000,0,0.9375,Low Risk,1
CU106,36500,2024-03-19,C006,Approved,Property,2023-07-21,24000,1,1.5208333333333333,Medium Risk,1
CU107,7200,2024-04-03,C007,Rejected,Auto,2023-08-14,11000,0,0.6545454545454545,Low Risk,1
CU108,49800,2024-04-18,C008,Approved,Health,2023-09-02,29000,1,1.717241379310345,Medium Risk,1
CU109,31000,2024-05-01,C009,Pending,Property,2023-10-11,21000,1,1.4761904761904763,Medium Risk,1
CU110,22500,2024-05-16,C010,Approved,Auto,2023-11-06,17000,1,1.3235294117647058,Medium Risk,1


In [0]:
#Transformation 4 : Monthly Claim Trends
df_transform= df_transform.withColumn("Claim_month", F.month("Claim_date"))
df_transform= df_transform.withColumn("Claim_year", F.year("Claim_date"))
Monthly_trends = df_transform.groupBy("Claim_year", "Claim_month").agg(F.sum("Claim_amount").alias("Monthly_total")).orderBy(F.col("Claim_year").desc(),F.col("Claim_month").desc())
Monthly_trends.display()

Claim_year,Claim_month,Monthly_total
2025,12,342700
2025,11,125000
2025,10,159000
2025,9,150600
2025,8,118300
2025,7,156000
2025,6,150500
2025,5,67700
2025,4,97200
2025,3,122500


In [0]:
#Transformations 5 : Loss Ratio Analysis
Loss_ratio = df_transform.groupBy("Claim_type").agg(
    (F.sum("Claim_amount") / F.sum("Premium")).alias("loss_ratio")
)
Loss_ratio.display()

Claim_type,loss_ratio
Auto,1.287513572204126
Health,1.461084529505582
Property,1.4088316467341306


In [0]:
# Save transformed dataset (Gold Zone) 
df_transform.write.format("delta") \
                   .mode("overwrite") \
                   .saveAsTable("insurance.insurance_gold")